# Notebook 03 — CAPM Benchmarking and Institutional Performance Metrics

## Objective

This notebook evaluates the development-period performance characteristics of
the seven eligible ETFs using SPY as the market benchmark and the US 3-month
Treasury yield as the risk-free rate.

The analysis uses the development sample ending on 31 December 2021 to avoid
look-ahead bias.

The main measures are:

- CAPM beta;
- CAPM expected return;
- Jensen's alpha;
- Sharpe ratio;
- Sortino ratio;
- Treynor ratio;
- Information ratio;
- maximum drawdown;
- Calmar ratio.

These measures provide a benchmark-relative and institutional perspective on
the individual assets before portfolio construction in Notebook 04.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

SIMPLE_RETURN_FILE = PROCESSED_DIR / "simple_returns.csv"
LOG_RETURN_FILE = PROCESSED_DIR / "log_returns.csv"
RF_FILE = EXTERNAL_DIR / "DGS3MO.csv"

print("Project root:")
print(PROJECT_ROOT.resolve())

print("\nSimple returns:")
print(SIMPLE_RETURN_FILE.resolve())
print("Exists:", SIMPLE_RETURN_FILE.exists())

print("\nLog returns:")
print(LOG_RETURN_FILE.resolve())
print("Exists:", LOG_RETURN_FILE.exists())

print("\nRisk-free file:")
print(RF_FILE.resolve())
print("Exists:", RF_FILE.exists())

Project root:
C:\quant_edge_round2

Simple returns:
C:\quant_edge_round2\data\processed\simple_returns.csv
Exists: True

Log returns:
C:\quant_edge_round2\data\processed\log_returns.csv
Exists: True

Risk-free file:
C:\quant_edge_round2\data\external\DGS3MO.csv
Exists: True


In [3]:
simple_returns = pd.read_csv(
    SIMPLE_RETURN_FILE,
    index_col=0,
    parse_dates=True
).sort_index()

log_returns = pd.read_csv(
    LOG_RETURN_FILE,
    index_col=0,
    parse_dates=True
).sort_index()

print("Simple returns shape:", simple_returns.shape)
print("Log returns shape:", log_returns.shape)

print("Start date:", simple_returns.index.min())
print("End date:", simple_returns.index.max())

simple_returns.head()

Simple returns shape: (5512, 7)
Log returns shape: (5512, 7)
Start date: 2004-11-19 00:00:00
End date: 2026-10-19 00:00:00


,SPY,EFA,EEM,AGG,GLD,XLE,XLK
2004-11-19,-0.011117,-0.006474,-0.014297,-0.002044,0.009013,0.011961,-0.015464
2004-11-22,0.004769,0.002933,0.003903,0.001561,0.003796,0.015943,0.001428
2004-11-23,0.001526,0.001949,0.001733,-0.000585,-0.004449,0.006223,-0.004278
2004-11-24,0.002370,0.004539,0.010647,0.000974,0.006704,0.008335,0.008115
2004-11-26,-0.000760,0.005811,0.016606,-0.002920,0.005327,0.004267,-0.001420


In [4]:
TRAIN_END = "2021-12-31"

simple_train = simple_returns.loc[:TRAIN_END].copy()
log_train = log_returns.loc[:TRAIN_END].copy()

assert not simple_train.empty
assert simple_train.shape == log_train.shape

print("Development sample start:", simple_train.index.min())
print("Development sample end:", simple_train.index.max())
print("Development observations:", len(simple_train))

Development sample start: 2004-11-19 00:00:00
Development sample end: 2021-12-31 00:00:00
Development observations: 4309


In [5]:
if not RF_FILE.exists():
    raise FileNotFoundError(
        f"DGS3MO.csv was not found here:\n{RF_FILE.resolve()}"
    )

rf_raw = pd.read_csv(RF_FILE)

print("Risk-free file loaded successfully.")
print("Columns:", list(rf_raw.columns))

rf_raw.head()

Risk-free file loaded successfully.
Columns: ['observation_date', 'DGS3MO']


,observation_date,DGS3MO
0,9/1/1981,17.01
1,9/2/1981,16.65
2,9/3/1981,16.96
3,9/4/1981,16.64
4,9/7/1981,NaN


In [6]:
rf_raw.columns = [
    str(col).strip()
    for col in rf_raw.columns
]

date_candidates = [
    "DATE",
    "Date",
    "date",
    "observation_date"
]

date_column = None

for col in date_candidates:
    if col in rf_raw.columns:
        date_column = col
        break

if date_column is None:
    raise KeyError(
        f"Could not identify the date column. "
        f"Available columns: {list(rf_raw.columns)}"
    )

if "DGS3MO" not in rf_raw.columns:
    raise KeyError(
        f"DGS3MO column not found. "
        f"Available columns: {list(rf_raw.columns)}"
    )

print("Date column detected:", date_column)

Date column detected:

 observation_date


In [7]:
rf_raw[date_column] = pd.to_datetime(
    rf_raw[date_column],
    errors="coerce"
)

rf_raw["DGS3MO"] = pd.to_numeric(
    rf_raw["DGS3MO"],
    errors="coerce"
)

rf_clean = (
    rf_raw[
        [date_column, "DGS3MO"]
    ]
    .dropna()
    .sort_values(date_column)
)

rf_clean.head()

,observation_date,DGS3MO
0,1981-09-01,17.01
1,1981-09-02,16.65
2,1981-09-03,16.96
3,1981-09-04,16.64
5,1981-09-08,16.54


In [8]:
rf_annual = (
    rf_clean
    .set_index(date_column)["DGS3MO"]
    / 100
)

rf_daily_raw = (
    (1 + rf_annual) ** (1 / 252)
) - 1

rf_daily_raw.head()

observation_date
1981-09-01    0.000624
1981-09-02    0.000611
1981-09-03    0.000622
1981-09-04    0.000611
1981-09-08    0.000608
Name: DGS3MO, dtype: float64

In [9]:
combined_index = (
    rf_daily_raw.index
    .union(simple_train.index)
)

rf_daily = (
    rf_daily_raw
    .reindex(combined_index)
    .sort_index()
    .ffill()
    .reindex(simple_train.index)
)

print("Aligned RF observations:", len(rf_daily))
print("Missing RF values:", rf_daily.isna().sum())

Aligned RF observations: 4309
Missing RF values: 0


In [10]:
TRADING_DAYS = 252

average_annual_rf_arithmetic = (
    rf_daily.mean()
    * TRADING_DAYS
)

average_annual_rf_effective = (
    (1 + rf_daily.mean()) ** TRADING_DAYS
) - 1

print(
    "Average annualized risk-free rate "
    "(arithmetic):"
)

print(
    f"{average_annual_rf_arithmetic:.4%}"
)

print(
    "\nAverage annualized risk-free rate "
    "(effective/compounded):"
)

print(
    f"{average_annual_rf_effective:.4%}"
)

Average annualized risk-free rate (arithmetic):
1.1775%

Average annualized risk-free rate (effective/compounded):
1.1844%


In [11]:
BENCHMARK = "SPY"

market_returns = simple_train[BENCHMARK]

market_excess = (
    market_returns
    - rf_daily
)

excess_returns = simple_train.sub(
    rf_daily,
    axis=0
)

print("Benchmark:", BENCHMARK)

Benchmark: SPY


In [12]:
def calculate_cagr(return_series):

    wealth = (
        1 + return_series
    ).cumprod()

    years = (
        return_series.index[-1]
        - return_series.index[0]
    ).days / 365.25

    return (
        wealth.iloc[-1] ** (1 / years)
    ) - 1

In [13]:
def calculate_drawdown(return_series):

    wealth = (
        1 + return_series
    ).cumprod()

    running_peak = wealth.cummax()

    drawdown = (
        wealth / running_peak
    ) - 1

    return drawdown

In [14]:
def calculate_downside_deviation(
    excess_return_series
):

    downside = np.minimum(
        excess_return_series,
        0
    )

    daily_downside = np.sqrt(
        np.mean(
            downside ** 2
        )
    )

    annual_downside = (
        daily_downside
        * np.sqrt(TRADING_DAYS)
    )

    return annual_downside

In [15]:
annual_rf = (
    rf_daily.mean()
    * TRADING_DAYS
)
annual_market_excess = (
    market_excess.mean()
    * TRADING_DAYS
)

metric_rows = []

drawdown_data = pd.DataFrame(
    index=simple_train.index
)

for ticker in simple_train.columns:

    asset_returns = simple_train[ticker]
    asset_excess = excess_returns[ticker]

    beta = (
        asset_excess.cov(market_excess)
        / market_excess.var()
    )

    capm_expected_return = (
        annual_rf
        + beta * annual_market_excess
    )

    annualized_return = (
        asset_returns.mean()
        * TRADING_DAYS
    )

    annualized_volatility = (
        asset_returns.std()
        * np.sqrt(TRADING_DAYS)
    )

    annualized_excess_return = (
        asset_excess.mean()
        * TRADING_DAYS
    )

    sharpe_ratio = (
        annualized_excess_return
        / annualized_volatility
        if annualized_volatility > 0
        else np.nan
    )

    downside_deviation = (
        calculate_downside_deviation(
            asset_excess
        )
    )

    sortino_ratio = (
        annualized_excess_return
        / downside_deviation
        if downside_deviation > 0
        else np.nan
    )
    treynor_ratio = (
        annualized_excess_return / beta
        if beta > 0.05
        else np.nan
    )

    jensen_alpha = (
        asset_excess.mean()
        - beta * market_excess.mean()
    ) * TRADING_DAYS

    active_returns = (
        asset_returns
        - market_returns
    )

    tracking_error = (
        active_returns.std()
        * np.sqrt(TRADING_DAYS)
    )

    information_ratio = (
        active_returns.mean()
        * TRADING_DAYS
        / tracking_error
        if tracking_error > 1e-12
        else np.nan
    )

    cagr = calculate_cagr(
        asset_returns
    )

    drawdown = calculate_drawdown(
        asset_returns
    )

    max_drawdown = drawdown.min()

    calmar_ratio = (
        cagr / abs(max_drawdown)
        if max_drawdown < 0
        else np.nan
    )

    drawdown_data[ticker] = drawdown

    metric_rows.append({
        "Ticker": ticker,
        "Annualized Mean Return": annualized_return,
        "CAGR": cagr,
        "Annualized Volatility": annualized_volatility,
        "Beta": beta,
        "CAPM Expected Return": capm_expected_return,
        "Jensen Alpha": jensen_alpha,
        "Sharpe Ratio": sharpe_ratio,
        "Downside Deviation": downside_deviation,
        "Sortino Ratio": sortino_ratio,
        "Treynor Ratio": treynor_ratio,
        "Tracking Error": tracking_error,
        "Information Ratio": information_ratio,
        "Maximum Drawdown": max_drawdown,
        "Calmar Ratio": calmar_ratio
    })

kpi_summary = pd.DataFrame(
    metric_rows
).set_index("Ticker")

kpi_summary

,Annualized Mean Return,CAGR,Annualized Volatility,Beta,CAPM Expected Return,Jensen Alpha,Sharpe Ratio,Downside Deviation,Sortino Ratio,Treynor Ratio,Tracking Error,Information Ratio,Maximum Drawdown,Calmar Ratio
Ticker,,,,,,,,,,,,,,
SPY,0.119341,0.105950,0.192376,1.000000,0.119341,0.000000,0.559144,0.137207,0.783970,0.107566,0.000000,NaN,-0.551894,0.191976
EFA,0.077998,0.055362,0.219176,1.023152,0.121832,-0.043833,0.302147,0.157520,0.420413,0.064725,0.096490,-0.428465,-0.610373,0.090702
EEM,0.110702,0.070292,0.292800,1.284773,0.149973,-0.039271,0.337867,0.203678,0.485704,0.077000,0.166226,-0.051971,-0.664343,0.105806
AGG,0.038966,0.038457,0.048709,-0.017447,0.009898,0.029067,0.558229,0.035824,0.759008,NaN,0.201690,-0.398509,-0.128352,0.299624
GLD,0.095099,0.081990,0.179955,0.039882,0.016065,0.079034,0.463023,0.127802,0.651973,NaN,0.257758,-0.094052,-0.455550,0.179980
XLE,0.099498,0.053421,0.307007,1.232478,0.144348,-0.044850,0.285735,0.219235,0.400131,0.071176,0.200067,-0.099184,-0.712628,0.074964
XLK,0.160539,0.147242,0.214523,1.021135,0.121615,0.038925,0.693466,0.150658,0.987433,0.145685,0.086305,0.477359,-0.530358,0.277627


## CAPM Interpretation

SPY has a beta of approximately one by construction.

EFA, EEM, XLE and XLK have positive market sensitivity, while AGG has near-zero
to slightly negative market beta and GLD has very low market beta.

Positive Jensen alpha indicates that realized return exceeded the CAPM-implied
return over the development sample, while negative alpha indicates
underperformance relative to CAPM.

AGG and GLD display low market sensitivity and therefore provide potential
diversification value despite lower standalone expected returns than the
highest-return equity assets.

In [16]:
institutional_summary = kpi_summary[
    [
        "Annualized Mean Return",
        "CAGR",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Sortino Ratio",
        "Treynor Ratio",
        "Jensen Alpha",
        "Information Ratio",
        "Maximum Drawdown",
        "Calmar Ratio"
    ]
].copy()

institutional_summary

,Annualized Mean Return,CAGR,Annualized Volatility,Sharpe Ratio,Sortino Ratio,Treynor Ratio,Jensen Alpha,Information Ratio,Maximum Drawdown,Calmar Ratio
Ticker,,,,,,,,,,
SPY,0.119341,0.105950,0.192376,0.559144,0.783970,0.107566,0.000000,NaN,-0.551894,0.191976
EFA,0.077998,0.055362,0.219176,0.302147,0.420413,0.064725,-0.043833,-0.428465,-0.610373,0.090702
EEM,0.110702,0.070292,0.292800,0.337867,0.485704,0.077000,-0.039271,-0.051971,-0.664343,0.105806
AGG,0.038966,0.038457,0.048709,0.558229,0.759008,NaN,0.029067,-0.398509,-0.128352,0.299624
GLD,0.095099,0.081990,0.179955,0.463023,0.651973,NaN,0.079034,-0.094052,-0.455550,0.179980
XLE,0.099498,0.053421,0.307007,0.285735,0.400131,0.071176,-0.044850,-0.099184,-0.712628,0.074964
XLK,0.160539,0.147242,0.214523,0.693466,0.987433,0.145685,0.038925,0.477359,-0.530358,0.277627


In [17]:
capm_summary = kpi_summary[
    [
        "Beta",
        "Annualized Mean Return",
        "CAPM Expected Return",
        "Jensen Alpha"
    ]
].copy()

capm_summary

,Beta,Annualized Mean Return,CAPM Expected Return,Jensen Alpha
Ticker,,,,
SPY,1.000000,0.119341,0.119341,0.000000
EFA,1.023152,0.077998,0.121832,-0.043833
EEM,1.284773,0.110702,0.149973,-0.039271
AGG,-0.017447,0.038966,0.009898,0.029067
GLD,0.039882,0.095099,0.016065,0.079034
XLE,1.232478,0.099498,0.144348,-0.044850
XLK,1.021135,0.160539,0.121615,0.038925


## Institutional Performance Interpretation

XLK exhibits the strongest standalone Sharpe and Sortino ratios during the
development period, but it also carries materially greater volatility and
drawdown risk than defensive assets.

AGG produces much lower volatility and maximum drawdown, making it valuable
from a capital-preservation perspective.

These results support the use of multi-asset portfolio construction rather
than simply selecting the asset with the highest historical return.

In [18]:
beta_min = min(
    0,
    kpi_summary["Beta"].min() - 0.1
)

beta_max = (
    kpi_summary["Beta"].max()
    + 0.1
)

beta_grid = np.linspace(
    beta_min,
    beta_max,
    200
)

sml_returns = (
    annual_rf
    + beta_grid
    * annual_market_excess
)

In [19]:
plt.figure(figsize=(9, 6))

plt.plot(
    beta_grid,
    sml_returns,
    label="Security Market Line"
)

for ticker in kpi_summary.index:

    x = kpi_summary.loc[
        ticker,
        "Beta"
    ]

    y = kpi_summary.loc[
        ticker,
        "Annualized Mean Return"
    ]

    plt.scatter(x, y)

    plt.text(
        x,
        y,
        f" {ticker}"
    )

plt.xlabel("Beta")
plt.ylabel("Annualized Mean Return")

plt.title(
    "CAPM Security Market Line"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "capm_security_market_line.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_16128\369732018.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
plt.figure(figsize=(9, 6))

for ticker in kpi_summary.index:

    x = kpi_summary.loc[
        ticker,
        "Annualized Volatility"
    ]

    y = kpi_summary.loc[
        ticker,
        "Annualized Mean Return"
    ]

    plt.scatter(x, y)

    plt.text(
        x,
        y,
        f" {ticker}"
    )

plt.xlabel(
    "Annualized Volatility"
)

plt.ylabel(
    "Annualized Mean Return"
)

plt.title(
    "Asset Risk-Return Map"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "asset_risk_return_map.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_16128\2188934677.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
plt.figure(figsize=(12, 6))

for ticker in drawdown_data.columns:

    plt.plot(
        drawdown_data.index,
        drawdown_data[ticker],
        label=ticker
    )

plt.axhline(
    0,
    linewidth=0.8
)

plt.xlabel("Date")
plt.ylabel("Drawdown")

plt.title(
    "Historical Drawdowns — Development Sample"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "asset_drawdowns_development.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_16128\3572284709.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
ranking_summary = pd.DataFrame(
    index=kpi_summary.index
)

ranking_summary["Sharpe Rank"] = (
    kpi_summary["Sharpe Ratio"]
    .rank(
        ascending=False,
        method="min"
    )
)

ranking_summary["Sortino Rank"] = (
    kpi_summary["Sortino Ratio"]
    .rank(
        ascending=False,
        method="min"
    )
)

ranking_summary["Calmar Rank"] = (
    kpi_summary["Calmar Ratio"]
    .rank(
        ascending=False,
        method="min"
    )
)

ranking_summary["Drawdown Rank"] = (
    kpi_summary[
        "Maximum Drawdown"
    ]
    .abs()
    .rank(
        ascending=True,
        method="min"
    )
)

ranking_summary

,Sharpe Rank,Sortino Rank,Calmar Rank,Drawdown Rank
Ticker,,,,
SPY,2.0,2.0,3.0,4.0
EFA,6.0,6.0,6.0,5.0
EEM,5.0,5.0,5.0,6.0
AGG,3.0,3.0,1.0,1.0
GLD,4.0,4.0,4.0,2.0
XLE,7.0,7.0,7.0,7.0
XLK,1.0,1.0,2.0,3.0


In [23]:
assert BENCHMARK in simple_train.columns

assert len(rf_daily) == len(simple_train)

assert rf_daily.isna().sum() == 0

assert np.isfinite(
    kpi_summary[
        "Annualized Volatility"
    ]
).all()

spy_beta = kpi_summary.loc[
    "SPY",
    "Beta"
]

print("SPY beta:", spy_beta)

assert np.isclose(
    spy_beta,
    1.0,
    atol=1e-8
)

print(
    "All Notebook 03 validation checks passed."
)

SPY beta: 1.0
All Notebook 03 validation checks passed.


In [24]:
rf_output = pd.DataFrame({
    "Risk Free Daily": rf_daily
})

rf_output.to_csv(
    PROCESSED_DIR
    / "risk_free_daily_aligned.csv"
)

In [25]:
kpi_summary.to_csv(
    TABLE_DIR
    / "asset_kpi_summary_development.csv"
)

capm_summary.to_csv(
    TABLE_DIR
    / "capm_summary_development.csv"
)

institutional_summary.to_csv(
    TABLE_DIR
    / "institutional_performance_summary_development.csv"
)

ranking_summary.to_csv(
    TABLE_DIR
    / "asset_metric_rankings_development.csv"
)

drawdown_data.to_csv(
    PROCESSED_DIR
    / "asset_drawdowns_development.csv"
)

print(
    "Notebook 03 outputs saved successfully."
)

Notebook 03 outputs saved successfully.


In [26]:
assert BENCHMARK in simple_train.columns

assert simple_train.shape == (
    4309,
    7
)

assert simple_train.index.max() == pd.Timestamp(
    "2021-12-31"
)

assert len(rf_daily) == len(simple_train)

assert rf_daily.isna().sum() == 0

assert (
    np.isfinite(
        kpi_summary[
            "Annualized Volatility"
        ]
    )
    .all()
)

assert np.isclose(
    kpi_summary.loc[
        "SPY",
        "Beta"
    ],
    1.0,
    atol=1e-8
)

assert (
    kpi_summary[
        "Annualized Volatility"
    ]
    > 0
).all()

assert (
    kpi_summary[
        "Maximum Drawdown"
    ]
    <= 0
).all()

print(
    "All Notebook 03 final "
    "validation checks passed."
)

All Notebook 03 final validation checks passed.
